# AIPerf Use Case 7 — Profiling Embedding Models

Profile an OpenAI-compatible **`/v1/embeddings`** endpoint with AIPerf: first with synthetic inputs of a controlled token length, then replaying a custom JSONL dataset, and compare latency/throughput between the two runs.

Based on the NVIDIA tutorial [Profiling Embedding Models with AIPerf](https://docs.nvidia.com/aiperf/dev/tutorials/model-endpoint-guides/profile-embedding-models-with-ai-perf). These use-case notebooks demonstrate AIPerf capabilities directly; the repo's per-scenario bash scripts remain the source of truth for the Model Selection / Sizing suites.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements and server setup](#2-requirements-and-server-setup)
3. [Configuration](#3-configuration)
4. [Test run — synthetic inputs](#4-test-run--synthetic-inputs)
5. [Test run — custom input file](#5-test-run--custom-input-file)
6. [Results analysis](#6-results-analysis)

## 1. Overview

Embedding models return one vector per request in a single non-streaming response, so profiling them looks different from the LLM use cases (UC1–UC5):

- **No token-level metrics** — TTFT and ITL don't exist here, and `--streaming` doesn't apply. There is no OSL either: the "output" is a fixed-size vector.
- The metrics that matter are **request latency** (avg/min/max/p50/p99) and **request throughput** (requests/sec), plus the **input sequence length** actually sent.
- `--endpoint-type embeddings` (with `--endpoint /v1/embeddings`) tells AIPerf to build OpenAI-style embeddings requests and parse the responses.

Why this matters for RAG deployments: query embedding sits in the critical path of every retrieval (a latency problem), while document embedding during ingest is a throughput problem — same endpoint, two different operating points. Profile both: short query-like inputs at moderate concurrency, and chunk-length inputs at higher concurrency.

## 2. Requirements and server setup

- `aiperf` CLI — the setup cell below installs it via pip if it isn't already on `PATH`. Pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible embeddings endpoint — any backend that serves `/v1/embeddings` works (vLLM, NIM, TEI in embedding mode, ...).
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).

Example server from the NVIDIA tutorial — vLLM serving `BAAI/bge-small-en-v1.5`:

```bash
docker run --gpus all -p 8000:8000 vllm/vllm-openai:latest \
  --model BAAI/bge-small-en-v1.5
```

Readiness check — repeat until it returns an embedding (HTTP 200):

```bash
curl -s localhost:8000/v1/embeddings \
  -H "Content-Type: application/json" \
  -d '{"model": "BAAI/bge-small-en-v1.5", "input": "test"}'
```

In [ ]:
import json
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

if shutil.which("aiperf") is None:
    print("aiperf CLI not found — installing into this kernel's environment ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "aiperf"], check=True)

aiperf_path = shutil.which("aiperf")
assert aiperf_path, "aiperf still not found after install — restart the kernel and rerun this cell."
print(f"aiperf found at: {aiperf_path}")
version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())


## 3. Configuration

| Flag | Meaning |
|---|---|
| `--endpoint-type embeddings` | Build/parse OpenAI-style embeddings requests |
| `--endpoint /v1/embeddings` | API path on the server |
| `--synthetic-input-tokens-mean` | Length (tokens) of the generated text samples |
| `--synthetic-input-tokens-stddev` | Sample lengths from a distribution instead of a fixed value |
| `--input-file` + `--custom-dataset-type single_turn` | Replay your own texts instead of synthetic ones |

Set `URL`, then run — the model name is read automatically from the endpoint's `/v1/models` listing (set `MODEL` manually only to override, e.g. when the server hosts several models). Defaults match the tutorial (20 requests at concurrency 4, ~100-token inputs) — small enough to finish in seconds against a single-GPU endpoint.

In [ ]:
# ---- Required ----------------------------------------------------------
URL = ""         # e.g. "http://localhost:8000"
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

# ---- Workload ----------------------------------------------------------
CONCURRENCY = 4
REQUEST_COUNT = 20
ISL_MEAN = 100            # synthetic input length (tokens)
ISL_STDDEV = 0

# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc7-embeddings"

assert URL, "Set URL in this cell."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")


## 4. Test run — synthetic inputs

AIPerf generates random text samples of roughly `ISL_MEAN` tokens and sends them as embeddings requests. Synthetic inputs make runs repeatable and comparable across models/backends; the trade-off is that lengths are uniform unless you set `ISL_STDDEV`. The synthetic and custom runs write to separate `-synthetic` / `-custom` artifact directories so the exports never overwrite each other.

In [ ]:
cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type embeddings
    --endpoint /v1/embeddings
    --concurrency {CONCURRENCY}
    --request-count {REQUEST_COUNT}
    --synthetic-input-tokens-mean {ISL_MEAN}
    --synthetic-input-tokens-stddev {ISL_STDDEV}
    --artifact-dir {OUTPUT_DIR}-synthetic
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")


## 5. Test run — custom input file

To measure with your actual content, provide a JSONL file — one embeddings request per line:

```json
{"texts": ["What is artificial intelligence?"]}
```

Input sequence lengths then vary with the real content instead of clustering around `ISL_MEAN`. The cell below writes the tutorial's five sample questions; replace `EMBEDDING_INPUTS` with your own texts (for RAG: use document chunks at your production chunk size, or real user queries).

In [ ]:
EMBEDDING_INPUTS = [
    "What is artificial intelligence?",
    "Explain machine learning.",
    "How do neural networks work?",
    "Define deep learning.",
    "What are transformers in AI?",
]

INPUT_FILE = f"{OUTPUT_DIR}-inputs.jsonl"  # relative to REPO_ROOT, like OUTPUT_DIR
input_path = REPO_ROOT / INPUT_FILE
input_path.parent.mkdir(parents=True, exist_ok=True)
input_path.write_text("\n".join(json.dumps({"texts": [t]}) for t in EMBEDDING_INPUTS) + "\n")
print(f"Wrote {len(EMBEDDING_INPUTS)} inputs to {input_path}")

cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type embeddings
    --endpoint /v1/embeddings
    --input-file {INPUT_FILE}
    --custom-dataset-type single_turn
    --request-count {len(EMBEDDING_INPUTS)}
    --artifact-dir {OUTPUT_DIR}-custom
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")


## 6. Results analysis

AIPerf writes the same export files as the LLM use cases (see the UC2 notebook for the full tour): `profile_export_aiperf.csv` (aggregated statistics in blank-line-separated sections), `profile_export_aiperf.json` (the same plus run config), and `profile_export.jsonl` (one record per request). For embeddings, the per-request statistics section carries **request latency** and **input sequence length**; the run-totals section carries **request throughput**. No TTFT/ITL/output-token rows appear.

In [ ]:
from io import StringIO

import pandas as pd


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


def stat(stats, pattern, col="avg"):
    m = stats[stats["Metric"].str.lower().str.contains(pattern)]
    return float(m[col].iloc[0]) if not m.empty else None


rows, full = [], {}
for run in ("synthetic", "custom"):
    d = REPO_ROOT / f"{OUTPUT_DIR}-{run}"
    csv_path = next(d.rglob("profile_export_aiperf.csv"), None)
    if csv_path is None:
        print(f"No profile_export_aiperf.csv under {d} — skipping the '{run}' run.")
        continue
    stats, totals = read_export(csv_path)
    full[run] = stats
    thr = totals[totals["Metric"].str.lower().str.contains("request throughput")]
    rows.append({
        "run": run,
        "request_latency_avg_ms": stat(stats, "request latency"),
        "request_latency_p50_ms": stat(stats, "request latency", "p50"),
        "request_latency_p99_ms": stat(stats, "request latency", "p99"),
        "input_seq_len_avg": stat(stats, "input sequence length"),
        "request_throughput_rps": float(thr["Value"].iloc[0]) if not thr.empty else None,
    })

display(pd.DataFrame(rows))

# Full per-request statistics for each completed run:
for run, stats in full.items():
    print(f"=== {run} run — full statistics ===")
    display(stats)


### Notes

- In the comparison table, the custom run's latency tracks its (shorter, varied) input lengths — comparing the two runs isolates how much input length drives embedding latency.
- Embedding servers batch aggressively, so throughput usually scales well with concurrency. Rerun the synthetic block at increasing `CONCURRENCY` values to find the knee — same idea as the UC1 concurrency sweep.
- A/B across embedding models or backends (vLLM vs. TEI vs. NIM): keep the command fixed and change only `MODEL`/`URL` — identical workload is what makes the numbers comparable.